# Open AI Embedding 사용하기
ChatGPT로 유명한 OpenAI에서 제공하는 임베딩 기법입니다. **유료**모델이며, OpenAI에서 api 키를 발급 받고 실습하시면 됩니다.

In [1]:
!pip install openai

# OpenAI API와의 통신을 간편하게 만들어 주며, ChatGPT와 같은 모델을 활용한 다양한 작업을 수행할 수 있음.

# openai 라이브러리의 주요 역할

# 모델 호출:
#     GPT-3, GPT-4 등의 언어 모델을 직접 호출하여 자연어 생성, 문서 요약, 번역, Q&A 등 다양한 텍스트 기반 작업을 수행할 수 있음
# API 호출 간편화:
#     OpenAI API를 쉽게 호출할 수 있는 인터페이스를 제공하여, 요청과 응답을 간단히 처리할 수 있도록 함.
# 다양한 작업 지원:
#     자연어 처리, 코드 생성, 이미지 생성, 텍스트 분석 등의 작업을 위해 모델과 통신 가능.
# 추론 결과 쉽게 확인:
#     응답을 다루기 쉽게 JSON 형식으로 반환하므로, 텍스트 응답을 읽고 처리하기가 용이.


In [2]:
from openai import OpenAI

API_KEY = "~~~~"

client = OpenAI(
    api_key=API_KEY
)

https://platform.openai.com/docs/guides/embeddings/embedding-models 참고

In [7]:
embedding = client.embeddings.create(
    model = "text-embedding-ada-002",
    input = "야 저기 차 온다"
)

len(embedding.data[0].embedding)


    # embedding = client.embeddings.create(...):

# client 객체를 사용하여 OpenAI API의 embeddings 엔드포인트(텍스트 데이터를 벡터로 변환하여 **임베딩(embedding)**을 생성할 수 있도록 하는 API 경로)를 호출하고 임베딩을 생성.
# create() 메소드는 텍스트를 벡터 형태로 변환하기 위한 함수로, 모델이 입력된 텍스트의 의미를 벡터로 표현.

    # 주요 파라미터:

# model='model_name': 임베딩을 생성할 때 사용할 모델의 이름을 지정.
# input='your_text': 벡터로 변환할 텍스트를 입력하는 부분.

    # embedding.data[0]:

# embedding 변수에는 OpenAI API로부터 반환된 응답 데이터가 저장.
# data는 임베딩 데이터가 포함된 리스트. 첫 번째 요소 [0]는 입력된 첫 번째 텍스트의 임베딩을 나타냄.

    # .embedding:

# embedding.data[0]은 임베딩과 관련된 메타데이터와 실제 임베딩 벡터를 포함하고 있으며, .embedding은 그 벡터 자체를 의미.

# text-embedding-ada-002 모델을 사용할 경우, 일반적으로 1536차원의 벡터를 반환합니다.

1536

# OpenAI 임베딩 수행

In [9]:
import pandas as pd

data = [
    "내일 차 타고 놀러가자",
    "지금 오는 버스는 어디서 오는 버스야?",
    "차 한잔 하면서 이야기 하시죠",
    "5차 공동구매! 오늘만 세일!",
    "홍차 녹차 중에 어떤 차가 좋으세요?",
]

df = pd.DataFrame(data, columns=['text'])
df

,text
0,내일 차 타고 놀러가자
1,지금 오는 버스는 어디서 오는 버스야?
2,차 한잔 하면서 이야기 하시죠
3,5차 공동구매! 오늘만 세일!
4,홍차 녹차 중에 어떤 차가 좋으세요?


In [10]:
def get_embedding(text):
    response = client.embeddings.create( input=text, model="text-embedding-ada-002")
    return response.data[0].embedding

In [11]:
df['embedding'] = df.apply(lambda row: get_embedding( row.text ), axis=1)
# List Comprehension 
# df['embedding'] = [get_embedding(text) for text in df['text']]


In [12]:
df

,text,embedding
0,내일 차 타고 놀러가자,"[0.015895135700702667, -0.011393633671104908, ..."
1,지금 오는 버스는 어디서 오는 버스야?,"[0.008037855848670006, -0.011817412450909615, ..."
2,차 한잔 하면서 이야기 하시죠,"[0.016478225588798523, -0.02388259395956993, 0..."
3,5차 공동구매! 오늘만 세일!,"[-0.017510101199150085, -0.002525998977944255,..."
4,홍차 녹차 중에 어떤 차가 좋으세요?,"[0.005494758952409029, -0.011675575748085976, ..."


# 유사도 구하기

In [13]:
import numpy as np

def cos_sim(A, B):
  return A @ B/(np.linalg.norm(A)*np.linalg.norm(B))

#  코사인 유사도는 두 벡터 간의 각도를 기반으로 유사도를 측정하는 방법으로 결과 값은 -1에서 1 사이의 값. 
#  1에 가까울수록 유사도가 높고, -1에 가까울수록 서로 반대 방향임을 나타내며, 0은 서로 직교하여 유사도가 없음을 의미 


def return_answer_candidate(df, query):
  '''
    df : 문장과 임베딩 벡터가 들어있는 데이터 프레임
    query : 질의할 문장
  '''
  query_embedding = get_embedding( query ) # query에 대한 임베딩 벡터 계산
  df["similarity"] = df.embedding.apply(lambda x: cos_sim(np.array(x), np.array(query_embedding))) # 계산한 query에 대한 임베딩 벡터와 text의 임베딩 벡터 간 유사도 계산
  
  # List Comprehension 
  # df["similarity"] = [cos_sim(np.array(x), np.array(query_embedding)) for x in df.embedding]
  # df["similarity"] = [ cos_sim( np.array(x) , np.array( query_embedding ) ) for x in df['embedding'] ]

  results_co = df.sort_values("similarity", ascending=False, ignore_index=True)
  return results_co.head(3)
# 쿼리와 유사도가 높은 문장들의 상위 3개를 찾아서 데이터프레임 형식으로 반환

In [14]:
sim_result = return_answer_candidate(df, '야 저기 차 온다')
sim_result

,text,embedding,similarity
0,내일 차 타고 놀러가자,"[0.015895135700702667, -0.011393633671104908, ...",0.896481
1,차 한잔 하면서 이야기 하시죠,"[0.016478225588798523, -0.02388259395956993, 0...",0.875385
2,지금 오는 버스는 어디서 오는 버스야?,"[0.008037855848670006, -0.011817412450909615, ...",0.848097


In [15]:
sim_result = return_answer_candidate(df, '예쁜 카페가고 싶어')
sim_result

,text,embedding,similarity
0,내일 차 타고 놀러가자,"[0.015895135700702667, -0.011393633671104908, ...",0.822717
1,차 한잔 하면서 이야기 하시죠,"[0.016478225588798523, -0.02388259395956993, 0...",0.812945
2,홍차 녹차 중에 어떤 차가 좋으세요?,"[0.005494758952409029, -0.011675575748085976, ...",0.812300
